In [1]:
!pip install youtube-transcript-api

In [2]:
!pip install -U langchain-text-splitters

  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.2
    Uninstalling langchain-text-splitters-1.1.2:
      Successfully uninstalled langchain-text-splitters-1.1.2


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()





/tmp/ipykernel_2081/1541534096.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [4]:
!pip install -U langchain-groq

In [5]:
from transformers import pipeline
from langchain_groq import ChatGroq
import torch

/var/www/html/myproject/GENAI-Series/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
!pip install accelerate

In [7]:
 !pip show youtube-transcript-api

Name: youtube-transcript-api
Version: 1.2.4
Summary: This is a python API which allows you to get the transcripts/subtitles for a given YouTube video. It also works for automatically generated subtitles, supports translating subtitles and it does not require a headless browser, like other selenium based solutions do!
Home-page: 
Author: Jonas Depoix
Author-email: jonas.depoix@web.de
License: MIT
Location: /var/www/html/myproject/GENAI-Series/venv/lib/python3.10/site-packages
Requires: defusedxml, requests
Required-by: 


In [8]:
!pip install langchain-text-splitters langchain-huggingface langchain-community faiss-cpu langchain-groq sentence-transformers

In [9]:

## Document Loader

yt_api = YouTubeTranscriptApi()

video_id = "SfOaZIGJ_gs"
transcript = yt_api.fetch(video_id=video_id)

transcript_text = " ".join([doc.text for doc in transcript])

## Text Splitting

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap= 50)
chunks = splitter.split_text(transcript_text)

## Call embedding models

embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

## Vector Store

vector_store = FAISS.from_texts(chunks,embeddings)

vector_store.similarity_search('Tell me about Ai')


## Retriever

retriever = vector_store.as_retriever(search_kwargs={"k":4})
retriever.invoke("How to start a company")


## Augmentation 
prompt = PromptTemplate(template=""" You are a helpful Assitant Annwer from the following context. If the context is insufficient then directly say I Dont know {context} Question {question} """, input_variables=['context','question'])

question = "How to start a compnay has been spoken in this video ? "

context = retriever.invoke(question)

context = " ".join(i.page_content for i in context)
final_prompt = prompt.invoke({'question':question, 'context':context})


## Generation

# model_id = "openai/gpt-oss-20b"
model_id = "openai/gpt-oss-120b"

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

response = llm.invoke(final_prompt)
print(response.content)







Loading weights: 100%|██████████| 103/103 [00:00<00:00, 792.54it/s]


The video highlights a few key pieces of advice for starting a company, drawn mainly from Paul Graham’s thinking and the speaker’s own experience:

1. **Make something people want** – This is the classic Y Combinator motto that Paul Graham gave to founders. It’s presented as the core, simplest‑yet‑most‑important instruction for building a successful startup.

2. **Take a long‑term horizon** – The speaker stresses the importance of thinking far ahead and being willing to wait. Their own company spent about **four and a half years** before launching a first product, using that time to experiment, iterate, and validate ideas.

3. **Think independently** – Rather than following trends, they focused on solving problems that mattered to them personally (“I started using it and realized that with my workflow I really needed this”). This independent, user‑centric approach helped them discover real demand.

So, according to the video, starting a company is about **identifying a genuine need (so